In [ ]:
import pandas as pd
import re
from pprint import pprint
import numpy as np


cgg_path = r'data/LAST_VERSION_OF_CGG3_20240410.xlsx'
cgg_df_orig = pd.read_excel(cgg_path, sheet_name='Sediment_Water', dtype=str)


General cleaning

In [2]:
cgg_df = cgg_df_orig.copy()
cgg_df = cgg_df.map(lambda x: x.strip() if isinstance(x, str) else x)
cgg_df = cgg_df.replace(['', 'nan'], np.nan)
cgg_df = cgg_df.dropna(how='all', axis='columns')

Combine all the unnamed columns into one

In [3]:
unnamed_cols = cgg_df.columns[cgg_df.columns.str.startswith('Unnamed')]
cgg_df['from_unnamed_cols'] = cgg_df[unnamed_cols].agg(lambda x: ' | '.join(x.dropna().astype(str)), axis=1).str.strip()
cgg_df = cgg_df.replace('', np.nan)
cgg_df = cgg_df.drop(columns=unnamed_cols)

# Cleaning lonitude and latititude columns

Standard cleaning


In [4]:
cgg_df['cleaned_lon'] = (cgg_df.Lon
                        .map(lambda x: re.sub(r'\s+', ' ', x), na_action='ignore')  # Removes any consecutive whitespaces
                        .map(lambda x: x.replace(',', '.'), na_action='ignore') # Standardize decimal point
                        .map(lambda x: x.replace(u'\xa0', u' '), na_action='ignore')  # Fix  weird unicode error
                        ) 

cgg_df['cleaned_lat'] = (cgg_df.Lat
                        .map(lambda x: re.sub(r'\s+', ' ', x), na_action='ignore')  # Removes any consecutive whitespaces
                        .map(lambda x: x.replace(',', '.'), na_action='ignore') # Standardize decimal point
                        .map(lambda x: x.replace(u'\xa0', u' '), na_action='ignore')  # Fix  weird unicode error
                        )   

Identify unique formats

In [5]:
lat_formats = (cgg_df.cleaned_lat
 .map(lambda x: re.sub(r'\d+', '12', x), na_action='ignore')  # Turns all n sized numbers into 123
 )

lat_formats.unique().tolist()

['+12.12',
 nan,
 '+12',
 '12',
 '12.12',
 '12 12.12 N',
 '12°12′N',
 'N12',
 '12 12 12.12 N',
 "W12 12' 12.12''",
 '12.12 N',
 '12deg12\'12.12"S',
 "N 12º12.12'",
 'N12.12',
 '12.12N',
 "12°12'N",
 "12°12'12''N",
 '12.12°',
 "12°12'12''",
 '12°12\'12"',
 '-12.12',
 'N 12.12',
 '12.12°N',
 '12.12˚N',
 '12˚12\'12.12" N',
 "12 12' 12'' N",
 'N12° 12\' 12.12"',
 '12.12° N',
 '12° 12.12',
 '12°12\'12.12"N',
 '12°12\'12"N',
 '12o12.12',
 '12° 12\' 12.12" N',
 '12°12’12”',
 '12°12\'12.12"',
 "12° 12' 12.12'' N",
 "12° 12'12.12'' N",
 '12° 12.12´S',
 '12° 12’ 12.12” N',
 '12°12’12.12’’N',
 '12° 12.12 N',
 '12°12\'12.12"S',
 '12°12‘12‘‘ N',
 "12' 12.12''",
 '12° 12’ 12.12”',
 '12°12\'12" N',
 '12° 12´12´´N',
 '12°12.12’N',
 '12° 12′ 12″ N']

In [6]:
lon_formats = (cgg_df.cleaned_lon
 .map(lambda x: re.sub(r'\d+', '12', x), na_action='ignore')  # Turns all n sized numbers into 123
 )

lon_formats.unique().tolist()

['-12.12',
 nan,
 '+12.12',
 '-12',
 '12.12',
 '12 12.12 E',
 "'+12.12",
 '12°12′W',
 'E12',
 '12 12 12.12 W',
 "N12 12' 12.12''",
 '12.12 E',
 '12deg12\'12.12"E',
 "E 12º12.12'",
 'E12.12',
 'E 12.12',
 'W12.12',
 '12.12W',
 "12°12'W",
 "12°12'12''E",
 '12',
 '12.12°',
 "12°12'12''",
 '12°12\'12"',
 '-12.12°W',
 '-12.12˚W',
 '12˚12\'12.12" W',
 "12 12' 12'' E",
 'W12° 12\' 12.12"',
 '12.12˚W',
 '12.12˚E',
 '12.12° E',
 '12.12 W',
 '12°12.12',
 '12°12\'12.12"E',
 '-12.12°',
 '12o12.12',
 '12° 12\' 12.12" W',
 '12.12°W',
 '-12.12˚E',
 '12°12\'12.12"',
 "12° 12' 12.12'' W",
 "12° 12'12.12'' W",
 "N12° 12' 12.12'' W",
 '12°12\'12.12"W',
 '12°12\'12.12"Ø',
 '12.12°E',
 '12°12\'12"W',
 "12° 12' 12.12'' V",
 '12°12\'12.12"V',
 '12°12.12`W',
 '12° 12’ 12.12”Ø.',
 '12° 12’ 12.12”Ø',
 '12°12’12.12’’ E',
 '12° 12 Ø',
 '12°12‘12‘‘ W',
 '12\' 12.12"',
 "12' 12.12''",
 '12° 12’ 12.12”',
 '12°12\'12" W',
 '12° 12´12´´ W',
 '12°12.12’W',
 '12° 12′ 12″ E']

### Convert to standard characters and symbols
The bad characters were found from the above step

In [7]:
cgg_df['cleaned_lon'] = (cgg_df.cleaned_lon
 .map(lambda x: re.sub(r"''|”|’’|‘‘|´´|″", '"', x), na_action='ignore')  # Replaces bad second chars with "
 .map(lambda x: re.sub(r"′|’|´|‘|`", "'", x), na_action='ignore')  # Replaces bad minute chars with '
 .map(lambda x: re.sub(r"(deg)|º|˚|o", "°", x), na_action='ignore')  # Replaces bad degree chars with °
 )

cgg_df['cleaned_lat'] = (cgg_df.cleaned_lat
 .map(lambda x: re.sub(r"''|”|’’|‘‘|´´|″", '"', x), na_action='ignore')  # Replaces bad second chars with "
 .map(lambda x: re.sub(r"′|’|´|‘", "'", x), na_action='ignore')  # Replaces bad minute chars with '
 .map(lambda x: re.sub(r"(deg)|º|˚|o", "°", x), na_action='ignore')  # Replaces bad degree chars with °
 )

Put direction indicators (N, S, -, +) in a seperate column

In [8]:
pd.set_option('future.no_silent_downcasting', True)

cgg_df['lon_direction'] = (cgg_df.cleaned_lon
                           .map(lambda x: re.sub(r"[^A-Za-zÆØÅæøå\-+]", '', x), na_action='ignore')
                           .replace(np.nan, '')              
                           )
#  Removes the direction from the cleaned_lon column, as now it is no longer needed
cgg_df.cleaned_lon = (cgg_df.cleaned_lon
                      .map(lambda x: re.sub(r"[A-Za-zÆØÅæøå\-+]", '', x), na_action='ignore')
                      .str.strip())

cgg_df['lat_direction'] = (cgg_df.cleaned_lat
                           .map(lambda x: re.sub(r"[^A-Za-zÆØÅæøå\-+]", '', x), na_action='ignore')
                           .replace(np.nan, '')  # This will make it easier later, when adding the direction back to the coordinate
                           )

#  Removes the direction from the cleaned_lat column, as now it is no longer needed
cgg_df.cleaned_lat = (cgg_df.cleaned_lat
                      .map(lambda x: re.sub(r"[A-Za-zÆØÅæøå\-+]", '', x), na_action='ignore')
                      .str.strip())

Identify unique formats again

In [9]:
from pprint import pprint
lon_formats = (cgg_df.cleaned_lon
 .map(lambda x: re.sub(r'\d+', '12', x), na_action='ignore')  # Turns all n sized numbers into 123
 )

lat_formats = (cgg_df.cleaned_lat
 .map(lambda x: re.sub(r'\d+', '12', x), na_action='ignore')  # Turns all n sized numbers into 123
 )

lon_formats = pd.Series(lon_formats.dropna().unique())
lat_formats = pd.Series(lat_formats.dropna().unique())

print('======Lon formats======')
print('With degree symbol')
pprint(lon_formats.dropna()[lon_formats.dropna().str.contains('°')].unique().tolist())
print('')
print('without degree symbol')
pprint(lon_formats.dropna()[~lon_formats.dropna().str.contains('°')].unique().tolist())

print()
print('======Lat formats======')
print('With degree symbol')
pprint(lat_formats.dropna()[lat_formats.dropna().str.contains('°')].unique().tolist())
print('')
print('without degree symbol')
pprint(lat_formats.dropna()[~lat_formats.dropna().str.contains('°')].unique().tolist())

======Lon formats======
With degree symbol
["12°12'",
 '12°12\'12.12"',
 "12°12.12'",
 '12°12\'12"',
 '12.12°',
 '12° 12\' 12.12"',
 '12°12.12',
 '12° 12\'12.12"',
 '12° 12\' 12.12".',
 '12° 12',
 '12° 12\'12"',
 '12° 12\' 12"']

without degree symbol
['12.12',
 '12',
 '12 12.12',
 "'12.12",
 '12 12 12.12',
 '12 12\' 12.12"',
 '12 12\' 12"',
 '12\' 12.12"']

======Lat formats======
With degree symbol
["12°12'",
 '12°12\'12.12"',
 "12°12.12'",
 '12°12\'12"',
 '12.12°',
 '12° 12\' 12.12"',
 '12° 12.12',
 '12°12.12',
 '12° 12\'12.12"',
 "12° 12.12'",
 '12° 12\'12"',
 '12° 12\' 12"']

without degree symbol
['12.12',
 '12',
 '12 12.12',
 '12 12 12.12',
 '12 12\' 12.12"',
 '12 12\' 12"',
 '12\' 12.12"']


From the above it seems safe to assume that if:
1. A coordinate contains 1 number with or without traling ° that the format is decimal degrees.
2. A coordinate contains 2 numbers seperated with with either a whitespace or ° or both that its formatted as degrees and minutes.
3. A coordinate contains 3 numbers, where first seperator is by ° or whitespace or both and second seperator is ' or whitespace or both, it is formatted as degree minute seconds.

We can formulate this using the following regular expressions:

In [10]:
dd_regex_lon = r'''^\d{1,3}(\.\d+)?(°| °)?$'''
dm_regex_lon = r'''^\d{1,3}( |° |°)\d{1,2}(\.\d+)?('| ')?$'''
dms_regex_lon = r'''^\d{1,3}( |° |°)\d{1,2}( |' |')\d{1,2}(\.\d+)?("| ")?$'''

dd_regex = r'''^\d{1,2}(\.\d+)?(°| °)?$'''
dm_regex = r'''^\d{1,2}( |° |°)\d{1,2}(\.\d+)?('| ')?$'''
dms_regex = r'''^\d{1,2}( |° |°)\d{1,2}( |' |')\d{1,2}(\.\d+)?("| ")?$'''


Checking the general formats lon

In [11]:
def check_format_lon(s: str):
    if re.match(dd_regex_lon, s):
        return 'DD'
    elif re.match(dm_regex_lon, s):
        return 'DM'
    elif re.match(dms_regex_lon, s):
        return 'DMS'
    else:
        return 'invalid format'

classified_formats = lon_formats.apply(check_format_lon)

lon_with_formats = pd.DataFrame({'lon': lon_formats, 'Format': classified_formats}).sort_values(by='Format')
lon_with_formats                           

,lon,Format
0,12.12,DD
1,12,DD
10,12.12°,DD
2,12 12.12,DM
16,12° 12,DM
4,12°12',DM
13,12°12.12,DM
8,12°12.12',DM
14,"12° 12'12.12""",DMS
12,"12° 12' 12.12""",DMS


Checking the general formats lat

In [12]:
def check_format_lat(s: str):
    if re.match(dd_regex, s):
        return 'DD'
    elif re.match(dm_regex, s):
        return 'DM'
    elif re.match(dms_regex, s):
        return 'DMS'
    else:
        return 'invalid format'

classified_formats = lat_formats.apply(check_format_lat)

lat_with_formats = pd.DataFrame({'Lat': lat_formats, 'Format': classified_formats}).sort_values(by='Format')
lat_with_formats                           

,Lat,Format
0,12.12,DD
9,12.12°,DD
1,12,DD
2,12 12.12,DM
3,12°12',DM
15,12° 12.12',DM
13,12°12.12,DM
7,12°12.12',DM
12,12° 12.12,DM
14,"12° 12'12.12""",DMS


Apply the same format identification to the actual data

In [13]:
cgg_df['lon_format'] = cgg_df.cleaned_lon.map(check_format_lon, na_action='ignore')
cgg_df['lat_format'] = cgg_df.cleaned_lat.map(check_format_lat, na_action='ignore')

Manual check that the lon format identification was done correctly

In [14]:
print('========Lon=========')
for ele in cgg_df['lon_format'].unique():
    print(ele)
    lon_formats = (cgg_df.query(f"lon_format == '{ele}'").cleaned_lon
        .map(lambda x: re.sub(r'\d', 'd', x), na_action='ignore')  # Turns all digits into 'd'
        .map(lambda x: re.sub(r'\.d+', '.dd', x), na_action='ignore')  # Turns all 'd's after a . into .dd 
    )

    lon_formats = pd.Series(lon_formats.dropna().unique())
    print(lon_formats)
    print()
    
print()
print('========Lat=========')
    
for ele in cgg_df['lat_format'].unique():
    print(ele)
    lat_formats = (cgg_df.query(f"lat_format == '{ele}'").cleaned_lat
        .map(lambda x: re.sub(r'\d', 'd', x), na_action='ignore')  # Turns all digits into 'd'
        .map(lambda x: re.sub(r'\.d+', '.dd', x), na_action='ignore')  # Turns all 'd's after a . into .dd 
    )

    lat_formats = pd.Series(lat_formats.dropna().unique())
    print(lat_formats)
    print()

========Lon=========
DD
0      dd.dd
1     ddd.dd
2       d.dd
3     dd.dd°
4    ddd.dd°
5      d.dd°
dtype: object

nan
Series([], dtype: object)

invalid format
0             dddddd
1              'd.dd
2           dddddddd
3            ddddddd
4         dd°dd'ddd"
5    dd° dd' dd.dd".
6         dd° dddddd
7         dd' dd.dd"
dtype: object

DM
0      dd dd.dd
1        dd°dd'
2     dd°dd.dd'
3       ddd°dd'
4     ddd°dd.dd
5    ddd°dd.dd'
dtype: object

DMS
0         dd dd dd.dd
1      ddd dd' dd.dd"
2        ddd dd dd.dd
3       ddd°dd'dd.dd"
4           dd°dd'dd"
5        dd°dd'dd.dd"
6          dd dd' dd"
7      dd° dd' dd.dd"
8         d°dd'dd.dd"
9       d° dd' dd.dd"
10      dd° dd' d.dd"
11       dd° dd'd.dd"
12           dd°dd'd"
13    ddd° dd' dd.dd"
14         dd° dd'dd"
15        dd° dd' dd"
dtype: object


========Lat=========
DD
0     dd.dd
1    dd.dd°
2      d.dd
dtype: object

nan
Series([], dtype: object)

invalid format
0          dddddddd
1            dddddd
2    dd

### Parse

Remove trailing non-numbers to make parsing easier

In [15]:
cgg_df.cleaned_lon = cgg_df.cleaned_lon.map(lambda x: re.sub(r"\D+$", '', x), na_action='ignore')  
cgg_df.cleaned_lat = cgg_df.cleaned_lat.map(lambda x: re.sub(r"\D+$", '', x), na_action='ignore')

##### Split lon data into degree minute and decimal second

If it looks good apply the splitting to the data

In [16]:
cgg_df['cleaned_lon_split'] = cgg_df.cleaned_lon.str.split(pat=r"[ °']+", regex=True)
cgg_df['cleaned_lat_split'] = cgg_df.cleaned_lat.str.split(pat=r"[ °']+", regex=True)

Manually inspect the splitting

In [17]:
print('============Lon============')
for ele in cgg_df['lon_format'].unique():
    print(ele)
    lon_formats = (cgg_df.query(f"lon_format == '{ele}'").cleaned_lon
        .map(lambda x: re.sub(r'\d', 'd', x), na_action='ignore')  # Turns all digits into 'd'
        .map(lambda x: re.sub(r'\.d+', '.dd', x), na_action='ignore')  # Turns all 'd's after a . into .dd 
    )

    lon_formats = pd.Series(lon_formats.dropna().unique())
    
    dms_formats_split = lon_formats.str.split(pat=r"[ °']+", regex=True)

    for raw, splits in zip(lon_formats, dms_formats_split):
        print(raw)
        print(splits)
        
        if ele == 'DD':
            assert len(splits) == 1
        if ele == 'DM':
            assert len(splits) == 2
        if ele == 'DMS':
            assert len(splits) == 3
    print()

print()
print('============Lat============')
for ele in cgg_df['lat_format'].unique():
    print(ele)
    lat_formats = (cgg_df.query(f"lat_format == '{ele}'").cleaned_lat
        .map(lambda x: re.sub(r'\d', 'd', x), na_action='ignore')  # Turns all digits into 'd'
        .map(lambda x: re.sub(r'\.d+', '.dd', x), na_action='ignore')  # Turns all 'd's after a . into .dd 
    )

    lat_formats = pd.Series(lat_formats.dropna().unique())
    
    dms_formats_split = lat_formats.str.split(pat=r"[ °']+", regex=True)

    for raw, splits in zip(lat_formats, dms_formats_split):
        print(raw)
        print(splits)
        
        if ele == 'DD':
            assert len(splits) == 1
        if ele == 'DM':
            assert len(splits) == 2
        if ele == 'DMS':
            assert len(splits) == 3
    print()

============Lon============
DD
dd.dd
['dd.dd']
ddd.dd
['ddd.dd']
d.dd
['d.dd']

nan

invalid format
dddddd
['dddddd']
'd.dd
['', 'd.dd']
dddddddd
['dddddddd']
ddddddd
['ddddddd']
dd°dd'ddd
['dd', 'dd', 'ddd']
dd° dd' dd.dd
['dd', 'dd', 'dd.dd']
dd° dddddd
['dd', 'dddddd']
dd' dd.dd
['dd', 'dd.dd']

DM
dd dd.dd
['dd', 'dd.dd']
dd°dd
['dd', 'dd']
dd°dd.dd
['dd', 'dd.dd']
ddd°dd
['ddd', 'dd']
ddd°dd.dd
['ddd', 'dd.dd']

DMS
dd dd dd.dd
['dd', 'dd', 'dd.dd']
ddd dd' dd.dd
['ddd', 'dd', 'dd.dd']
ddd dd dd.dd
['ddd', 'dd', 'dd.dd']
ddd°dd'dd.dd
['ddd', 'dd', 'dd.dd']
dd°dd'dd
['dd', 'dd', 'dd']
dd°dd'dd.dd
['dd', 'dd', 'dd.dd']
dd dd' dd
['dd', 'dd', 'dd']
dd° dd' dd.dd
['dd', 'dd', 'dd.dd']
d°dd'dd.dd
['d', 'dd', 'dd.dd']
d° dd' dd.dd
['d', 'dd', 'dd.dd']
dd° dd' d.dd
['dd', 'dd', 'd.dd']
dd° dd'd.dd
['dd', 'dd', 'd.dd']
dd°dd'd
['dd', 'dd', 'd']
ddd° dd' dd.dd
['ddd', 'dd', 'dd.dd']
dd° dd'dd
['dd', 'dd', 'dd']
dd° dd' dd
['dd', 'dd', 'dd']


============Lat============
DD
dd.dd
['dd.dd']


Test if the different formats have correct number of elementes in the split lists and that each element only contains numbers

In [18]:

# =================Lon============================
for i, row in cgg_df[['lon_format', 'cleaned_lon_split']].iterrows():
    split_lst = row.cleaned_lon_split
    lon_format = row.lon_format
    
    if lon_format == 'DD':
        assert len(split_lst) == 1
    if lon_format == 'DM':
        assert len(split_lst) == 2
    if lon_format == 'DMS':
        assert len(split_lst) == 3
        
    if isinstance(split_lst, list) and lon_format != 'invalid format':
        for ele in split_lst:
            try:
                float(ele)
            except ValueError:
                raise Exception(f'bad list: {split_lst}')
            
# =================Lat============================
            
for i, row in cgg_df[['lat_format', 'cleaned_lat_split']].iterrows():
    split_lst = row.cleaned_lat_split
    lat_format = row.lat_format
    
    if lat_format == 'DD':
        assert len(split_lst) == 1
    if lat_format == 'DM':
        assert len(split_lst) == 2
    if lat_format == 'DMS':
        assert len(split_lst) == 3
    if isinstance(split_lst, list):
        for ele in split_lst:
            try:
                float(ele)
            except ValueError:
                raise Exception(f'bad list: {split_lst}')

Convert direction to plus and minus

In [19]:
def convert_direction_lon(row):
    direction = str(row.lon_direction)
    if direction == 'nan':
        return np.nan
    direction = re.sub(r'[EeØø]', '+', direction)
    direction = re.sub(r'[WwVv]', '-', direction)
    direction = direction.strip()
    if bool(re.match(r'^(\-|\+)$', direction)) or direction == '':
        return direction
    else:
        
        return 'invalid direction'

def convert_direction_lat(row):
    direction = str(row.lat_direction)
    if direction == 'nan':
        return np.nan
    direction = re.sub(r'[Nn]', '+', direction)
    direction = re.sub(r'[Ss]', '-', direction)
    direction = direction.strip()
    if bool(re.match(r'^(\-|\+)$', direction)) or direction == '':
        return direction
    else:
        
        return 'invalid direction'

cgg_df['converted_lon_direction'] = cgg_df.apply(convert_direction_lon, axis=1)
cgg_df['converted_lat_direction'] = cgg_df.apply(convert_direction_lat, axis=1)
print(cgg_df['converted_lat_direction'].unique())
print(cgg_df['converted_lon_direction'].unique())

['+' '' 'invalid direction' '-']
['-' '' '+' 'invalid direction']


Convert Split data into decimal degrees and add converted direction lon

In [20]:
def add_direction(row):
    direction = str(row.converted_lon_direction)
    coord = row.converted_lon
    if not direction == 'invalid direction':
        return float(str(direction) + str(coord))
    else:
        return np.nan

def convert_dd(lst):
    assert len(lst) == 1
    return float(lst[0])

def convert_dm(lst):
    assert len(lst) == 2
    degrees, minutes = float(lst[0]), float(lst[1])
    
    return degrees + (minutes/60)

def convert_dms(lst):
    assert len(lst) == 3
    degrees, minutes, seconds = float(lst[0]), float(lst[1]), float(lst[2])
    
    return degrees + (minutes/60) + (seconds/3600)

def convert_to_dd(row):
  
    lon_format = row.lon_format 
    split_lst = row.cleaned_lon_split
    
    if lon_format == 'DD':
        result = convert_dd(split_lst)
    elif lon_format == 'DM':
        result = convert_dm(split_lst)
    elif lon_format == 'DMS':
        result = convert_dms(split_lst)
    else:
        return np.nan
    return result
    
cgg_df['converted_lon'] = cgg_df.apply(convert_to_dd, axis=1)
cgg_df['converted_lon'] = cgg_df.apply(add_direction, axis=1)

assert cgg_df.converted_lon.dropna().astype(str).apply(lambda x: bool(re.fullmatch(r'^\-?\d{1,3}\.\d*$', x))).all()

print()
print('The following directions were marked as invalid:')
print(cgg_df.query('converted_lon_direction == "invalid direction"')['lon_direction'].unique())


The following directions were marked as invalid:
['N' '-W' '-E' 'NW']


Same as above but for lat

In [21]:
def add_direction(row):
    direction = str(row.converted_lat_direction)
    coord = row.converted_lat
    if not direction == 'invalid direction':
        return float(str(direction) + str(coord))
    else:
        return np.nan

def convert_dd(lst):
    assert len(lst) == 1
    return float(lst[0])

def convert_dm(lst):
    assert len(lst) == 2
    degrees, minutes = float(lst[0]), float(lst[1])
    
    return degrees + (minutes/60)

def convert_dms(lst):
    assert len(lst) == 3
    degrees, minutes, seconds = float(lst[0]), float(lst[1]), float(lst[2])
    
    return degrees + (minutes/60) + (seconds/3600)

def convert_to_dd(row):
    lat_format = row.lat_format 
    split_lst = row.cleaned_lat_split
    if lat_format == 'DD':
        return convert_dd(split_lst)
    elif lat_format == 'DM':
        return convert_dm(split_lst)
    elif lat_format == 'DMS':
        return convert_dms(split_lst)
    else:
        return np.nan
    
cgg_df['converted_lat'] = cgg_df.apply(convert_to_dd, axis=1)
cgg_df['converted_lat'] = cgg_df.apply(add_direction, axis=1)

assert cgg_df.converted_lat.dropna().astype(str).apply(lambda x: bool(re.fullmatch(r'^\-?\d{1,2}\.\d*$', x))).all()

print()
print('The following directions were marked as invalid:')
print(cgg_df.query('converted_lat_direction == "invalid direction"')['lat_direction'].unique())


The following directions were marked as invalid:
['W']


Manually inspect the data to verify lon

In [22]:
cgg_df.query('lon_format == "DMS"')[['Lon', 'cleaned_lon', 'converted_lon', 'lon_format', 'lon_direction', 'converted_lon_direction']].sample(10)

,Lon,cleaned_lon,converted_lon,lon_format,lon_direction,converted_lon_direction
17509,"13°58'0"" W",13°58'0,-13.966667,DMS,W,-
17555,"13°58'0"" W",13°58'0,-13.966667,DMS,W,-
10632,"8°27'57.88""E",8°27'57.88,8.466078,DMS,E,+
2561,N056 20' 15.66'',056 20' 15.66,NaN,DMS,N,invalid direction
17622,21° 37´27´´ W,21° 37'27,-21.624167,DMS,W,-
12328,22° 44'7.4832'' W,22° 44'7.4832,-22.735412,DMS,W,-
15501,"12°35'55.175""",12°35'55.175,12.598660,DMS,,
10486,"46°53'29.17""E",46°53'29.17,46.891436,DMS,E,+
15529,"12°35'55.175""",12°35'55.175,12.598660,DMS,,
12657,"19°10'13.0""W",19°10'13.0,-19.170278,DMS,W,-


In [23]:
cgg_df.query('lon_format == "DM"')[['Lon', 'cleaned_lon', 'converted_lon', 'lon_format', 'lon_direction']].sample(10)

,Lon,cleaned_lon,converted_lon,lon_format,lon_direction
5021,113°27'W,113°27,-113.450000,DM,W
5120,113°27'W,113°27,-113.450000,DM,W
10314,015°11.391,015°11.391,15.189850,DM,
3677,E 93º24.429',93°24.429,93.407150,DM,E
2045,37°55′W,37°55,-37.916667,DM,W
10383,015°11.391,015°11.391,15.189850,DM,
13260,071°18.180`W,071°18.180,-71.303000,DM,W
5134,113°27'W,113°27,-113.450000,DM,W
13271,071°18.180`W,071°18.180,-71.303000,DM,W
13263,071°18.180`W,071°18.180,-71.303000,DM,W


In [24]:
cgg_df.query('lon_format == "DD"')[['Lon', 'cleaned_lon', 'converted_lon', 'lon_format', 'lon_direction']].sample(10)

,Lon,cleaned_lon,converted_lon,lon_format,lon_direction
20639,9.5142654°,9.5142654,9.514265,DD,
14713,120.169786,120.169786,120.169786,DD,
17236,59.74806,59.74806,59.748060,DD,
15383,95.22767231,95.22767231,95.227672,DD,
7430,-45.367166˚W,45.367166,NaN,DD,-W
12055,-82.373783,82.373783,-82.373783,DD,-
19934,"-17,700213",17.700213,-17.700213,DD,-
11543,14.9320˚W,14.9320,-14.932000,DD,W
18308,-2.389444,2.389444,-2.389444,DD,-
9479,14.3875° E,14.3875,14.387500,DD,E


In [25]:
cgg_df.query('lon_format == "invalid format"')[['Lon', 'cleaned_lon', 'converted_lon', 'lon_format', 'lon_direction']].sample(10)

,Lon,cleaned_lon,converted_lon,lon_format,lon_direction
16305,"28' 15.24""",28' 15.24,NaN,invalid format,
12619,"21°55'464""W",21°55'464,NaN,invalid format,W
5401,51888806,51888806,NaN,invalid format,
16906,19' 29.70'',19' 29.70,NaN,invalid format,
1995,-50201302,50201302,NaN,invalid format,-
20584,19' 29.70'',19' 29.70,NaN,invalid format,
20806,"29' 32.522""",29' 32.522,NaN,invalid format,
16252,"28' 15.24""",28' 15.24,NaN,invalid format,
16278,"28' 15.24""",28' 15.24,NaN,invalid format,
16283,"28' 15.24""",28' 15.24,NaN,invalid format,


Manually inspect the data to verify lat

In [26]:
cgg_df.query('lat_format == "DMS"')[['Lat', 'cleaned_lat', 'converted_lat', 'lat_format', 'lat_direction']].sample(10)

,Lat,cleaned_lat,converted_lat,lat_format,lat_direction
12361,66° 3' 40.0608'' N,66° 3' 40.0608,66.061128,DMS,N
10608,"71°00'15""N",71°00'15,71.004167,DMS,N
10602,"71°00'15""N",71°00'15,71.004167,DMS,N
12295,"55°41'17.052""",55°41'17.052,55.688070,DMS,
20685,"55°40'53.163""",55°40'53.163,55.681434,DMS,
12645,"63°57'22""N",63°57'22,63.956111,DMS,N
10668,"71°00'15""N",71°00'15,71.004167,DMS,N
11960,"50° 51' 49.53"" N",50° 51' 49.53,50.863758,DMS,N
2248,"85 53 49,66 N",85 53 49.66,85.897128,DMS,N
10449,"34°35'2.19""N",34°35'2.19,34.583942,DMS,N


In [27]:
cgg_df.query('lat_format == "DM"')[['Lat', 'cleaned_lat', 'converted_lat', 'lat_format', 'lat_direction']].sample(10)

,Lat,cleaned_lat,converted_lat,lat_format,lat_direction
11291,50o31.331,50°31.331,50.522183,DM,
3732,N 73º21.025',73°21.025,73.350417,DM,N
5131,52°04'N,52°04,52.066667,DM,N
13296,23° 03.139´S,23° 03.139,-23.052317,DM,S
3680,N 73º09.387',73°09.387,73.156450,DM,N
3630,N 73º21.025',73°21.025,73.350417,DM,N
5003,52°04'N,52°04,52.066667,DM,N
5134,52°04'N,52°04,52.066667,DM,N
2061,65°41′N,65°41,65.683333,DM,N
11298,50o31.331,50°31.331,50.522183,DM,


In [28]:
cgg_df.query('lat_format == "DD"')[['Lat', 'cleaned_lat', 'converted_lat', 'lat_format', 'lat_direction']].sample(10)

,Lat,cleaned_lat,converted_lat,lat_format,lat_direction
14839,30.512656,30.512656,30.512656,DD,
14637,30.512656,30.512656,30.512656,DD,
20124,"66,213895",66.213895,66.213895,DD,
13863,30.512615,30.512615,30.512615,DD,
6303,54.967521,54.967521,54.967521,DD,
4896,52.21N,52.21,52.210000,DD,N
6381,N66.3898,66.3898,66.389800,DD,N
4830,52.21N,52.21,52.210000,DD,N
20142,"66,50076",66.50076,66.500760,DD,
11133,29.3819,29.3819,29.381900,DD,


In [29]:
cgg_df.query('lat_format == "invalid format"')[['Lat', 'cleaned_lat', 'converted_lat', 'lat_format', 'lat_direction']].sample(10)

,Lat,cleaned_lat,converted_lat,lat_format,lat_direction
3131,W116 11' 47.538'',116 11' 47.538,NaN,invalid format,W
2637,W120 57' 37.002'',120 57' 37.002,NaN,invalid format,W
318,+50483248,50483248,NaN,invalid format,+
5416,55111433,55111433,NaN,invalid format,
2008,64398201,64398201,NaN,invalid format,
9599,116.026184,116.026184,NaN,invalid format,
9525,116.090474,116.090474,NaN,invalid format,
2705,W120 57' 37.002'',120 57' 37.002,NaN,invalid format,W
3243,W120 57' 37.002'',120 57' 37.002,NaN,invalid format,W
2765,W116 11' 47.538'',116 11' 47.538,NaN,invalid format,W


### Stats

Number of rows where either lat or lon is invalid or missing

In [30]:
len(cgg_df.query("lat_format == 'invalid format' | converted_lat_direction == 'invalid direction' | lon_format == 'invalid format' | converted_lon_direction == 'invalid direction' | Lat.isnull() | Lon.isnull()"))

9306

In [31]:
cgg_df['data_cleaning_note'] = None
cgg_df.loc[
    (cgg_df["lat_format"] == "invalid format") |
    (cgg_df["converted_lat_direction"] == "invalid direction") |
    (cgg_df["lon_format"] == "invalid format") |
    (cgg_df["converted_lon_direction"] == "invalid direction") |
    (cgg_df["Lat"].isna()) |
    (cgg_df["Lon"].isna()),
    "data_cleaning_note"
] = "Needs human validation"

Save the file

In [32]:
cgg_df.to_csv(r'data/cgg_clean_lat_lon.tsv', sep='\t', index=False)